# OpenAI SDK - Streaming

When using streams with the SDK you don't receive raw lines, as with httpx, but you receive typed SSE events already parsed.

There are mainly three kind of events:
- `response.output_text.delta`
    - contains a delta piece of the message so that you can compose in realtime
- `response.completed`
    - contains `event.response` with the whole generated message and the usage info
- `response.failed` / `response.error` 

Gotchas (hard-learned):
- When using `client.responses.create(...stream=True)` or `client.responses.stream()` you need a `with` statement because they return a streaming response that holds an open HTTP connection, so you need to ensure that this will be closed properly even if an error is raised or you break the loop.
    - Non-streamed `create()` doesn't need `with` because the response is parsed inside the function and so the object returned doesn't hold an HTTP connection 
- When using structured output you can access the accumulated partial result using `event.snapshot` but:
    - you have to parse the result yourself using `pydantic_core.from_json(..., allow_partial=True)`
    - __you cannot assume that the last element is complete__

## Usage

### unstructured output sync

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

# load .env vars to environment variables
load_dotenv() 

# example without pydantic models, just using the raw json output from the model
# unlike httpx, here you'll receive usage information automatically
# you need to use with to ensure the stream is closed properly even if an exception is raised or the loop is broken
client = OpenAI()

with client.responses.create(
    model="gpt-5-nano",
    instructions="You are a helpful friend.",
    input="List three kinds of label adhesives.",
    reasoning={"effort": "minimal"},
    store=False,
    stream=True, # !!
) as stream:
    
    for event in stream:
        if event.type == "response.output_text.delta":
            print(f"${event.delta}", end="", flush=True)
        elif event.type == "response.completed":
            print(f"\nevent: {event.type} - {event}")
            print(f"usage: {event.response.usage}")
        # else:
        #     print(f"event: {event.type} - {event}")

True


### unstructured output async

In [ ]:
from openai import AsyncOpenAI

# same as synchronous example, but using async/await
# differences:
# - you need to use `await` when calling the `client.responses.create()` method
# - you need to use `async for` when iterating over the stream
client = AsyncOpenAI()

async with client.responses.create(
    model="gpt-5-nano",
    instructions="You are a helpful friend.",
    input="List three kinds of label adhesives.",
    reasoning={"effort": "minimal"},
    store=False,
    stream=True, # !!
) as stream:
    
    async for event in stream:
        if event.type == "response.output_text.delta":
            print(f"${event.delta}", end="", flush=True)
        elif event.type == "response.completed":
            print(f"\nevent: {event.type} - {event}")
            print(f"usage: {event.response.usage}")

### structured output async

In [43]:
import json
from openai import AsyncOpenAI
from pydantic import BaseModel
from pydantic_core import from_json

payment_descriptions = [
    { "id": 1, "payment_description": "PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026" },
    { "id": 2, "payment_description": "SALDO FATTURA N. 6549 DEL 30 04 2026" }
]
class Invoice(BaseModel):
    invoice_number: int | None
    date: str | None
class Classification(BaseModel):
    id: int
    payment_description: str
    invoices: list[Invoice]
    confidence: float
    notes: str | None
class Batch(BaseModel):
    results: list[Classification]

client = AsyncOpenAI()
i = 0

# with structured output you use .stream()
async with client.responses.stream(
    model="gpt-5-nano",
    instructions="you are a classifier of Italian bank credit transfer descriptions (causali bonifico).",
    input=json.dumps(payment_descriptions, ensure_ascii=False, indent=2),
    text_format=Batch,
    reasoning={"effort": "minimal"},
    store=False,
) as stream:
    async for event in stream:
        if event.type == "response.output_text.delta":
            if i % 20 == 0: # sampling the snapshots to avoid too many prints
                partial_structured_output = from_json(event.snapshot, allow_partial=True)
                print(f"partial output - delta #{i}: {partial_structured_output}")
            i += 1
    print(f"total deltas: {i}")

    final = await stream.get_final_response()
    print(f"final output: {final.output_parsed}")
    print(f"usage - input: {final.usage.input_tokens} - output: {final.usage.output_tokens} - total: {final.usage.total_tokens}")

partial output - delta #0: {}
partial output - delta #20: {'results': [{'id': 1}]}
partial output - delta #40: {'results': [{'id': 1, 'payment_description': 'PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026', 'invoices': [{}]}]}
partial output - delta #60: {'results': [{'id': 1, 'payment_description': 'PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026', 'invoices': [{'invoice_number': 999, 'date': '2026-01-31'}], 'confidence': 0.55}]}
partial output - delta #80: {'results': [{'id': 1, 'payment_description': 'PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026', 'invoices': [{'invoice_number': 999, 'date': '2026-01-31'}], 'confidence': 0.55, 'notes': None}, {'id': 2}]}
partial output - delta #100: {'results': [{'id': 1, 'payment_description': 'PAG. 2 ACC. FATT. N. 999 DEL 31 01 2026', 'invoices': [{'invoice_number': 999, 'date': '2026-01-31'}], 'confidence': 0.55, 'notes': None}, {'id': 2, 'payment_description': 'SALDO FATTURA N. 6549 DEL 30 04 2026', 'invoices': [{'invoice_number': 6549}]}]}
total deltas: 120
final 